In [ ]:
%%capture
%%bash
# 1. Instalar espeak-ng en el sistema operativo para convertir texto en fonemas
apt-get update -qq && apt-get install -y -qq espeak-ng > /dev/null 2>&1

# 2. Clonar el repositorio oficial de StyleTTS 2
git clone -q https://github.com/yl4579/StyleTTS2.git
cd StyleTTS2

# 3. Instalar las librerías de Python necesarias (silenciando los mensajes de pip)
pip install -q SoundFile torchaudio munch pydub pyyaml librosa nltk matplotlib accelerate transformers einops einops-exts tqdm typing-extensions git+https://github.com/resemble-ai/monotonic_align.git phonemizer > /dev/null 2>&1

# 4. Crear carpeta y descargar los pesos preentrenados multispeaker (LibriTTS) desde HuggingFace
mkdir -p Models/LibriTTS
wget -q https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/main/Models/LibriTTS/epochs_2nd_00020.pth -O Models/LibriTTS/epochs_2nd_00020.pth
wget -q https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/main/Models/LibriTTS/config.yml -O Models/LibriTTS/config.yml

In [ ]:
import glob
import os
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

# 1. Localizar speaker-info.txt en /kaggle/input
vctk_files = glob.glob("/kaggle/input/**/speaker-info.txt", recursive=True)
assert len(vctk_files) > 0, "No se encontró speaker-info.txt."
vctk_root = os.path.dirname(vctk_files[0])

# 2. Parsear speaker-info.txt línea a línea (evita errores por espacios en la columna REGION)
rows = []
with open(vctk_files[0], "r", encoding="utf-8") as f:
    for line in f.readlines()[1:]:  # saltar cabecera
        parts = line.strip().split()
        if len(parts) >= 4:
            raw_id = parts[0]
            spk_id = raw_id if raw_id.startswith("p") else f"p{raw_id}"
            rows.append({
                "ID": spk_id,
                "AGE": parts[1],
                "GENDER": parts[2],
                "ACCENTS": parts[3],
            })

df_spk = pd.DataFrame(rows)
df_spk["AGE"] = pd.to_numeric(df_spk["AGE"], errors="coerce")
df_spk = df_spk.dropna(subset=["AGE", "GENDER"]).reset_index(drop=True)

# 3. Localizar la carpeta base de AUDIOS (.wav o .flac), evitando la carpeta 'txt'
audio_candidates = [
    os.path.dirname(p)
    for p in glob.glob(f"{vctk_root}/**/p225/*.*", recursive=True)
    if p.lower().endswith((".wav", ".flac")) and "/txt/" not in p.lower()
]
assert len(audio_candidates) > 0, (
    f"No se encontraron archivos de audio dentro de {vctk_root}"
)

audio_base = os.path.dirname(audio_candidates[0])
print(f"Carpeta de audios detectada: {audio_base}")

# 4. Filtrar solo locutores que realmente tengan carpeta con audios
available_spks = {
    d
    for d in os.listdir(audio_base)
    if os.path.isdir(os.path.join(audio_base, d))
    and len(glob.glob(os.path.join(audio_base, d, "*.*"))) > 0
}
df_spk = df_spk[df_spk["ID"].isin(available_spks)].reset_index(drop=True)
assert len(df_spk) > 0, "0 locutores tras filtrar."

# 5. Partición estratificada 80% train / 20% test disjunta por locutor
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in splitter.split(df_spk, df_spk["GENDER"]):
    df_spk["split"] = "train"
    df_spk.loc[test_idx, "split"] = "test"

# 6. Guardar el CSV con la división
os.makedirs("/kaggle/working/data_splits", exist_ok=True)
df_spk.to_csv("/kaggle/working/data_splits/vctk_speakers_split.csv", index=False)

print(f"VCTK listo. Total locutores válidos: {len(df_spk)}")
print(df_spk.groupby(["split", "GENDER"])["AGE"].agg(["count", "mean", "std"]))

In [ ]:
%%capture
import sys
sys.path.append('/kaggle/working/StyleTTS2')

import torch

# Parche de compatibilidad para PyTorch 2.6+ (permite cargar los checkpoints antiguos de StyleTTS 2)
_orig_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

import torchaudio
import librosa
import yaml
import phonemizer
import numpy as np
import soundfile as sf
from collections import OrderedDict
import nltk
from nltk.tokenize import word_tokenize
from models import *
from utils import *
from text_utils import TextCleaner
from Utils.PLBERT.util import load_plbert

# Descargar tokenizador de frases
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Seleccionar GPU si está disponible
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. Cargar el archivo de configuración del modelo
config_path = "/kaggle/working/StyleTTS2/Models/LibriTTS/config.yml"
model_path = "/kaggle/working/StyleTTS2/Models/LibriTTS/epochs_2nd_00020.pth"
config = yaml.safe_load(open(config_path))

# 2. Cargar los módulos auxiliares preentrenados (alineador de texto, extractor de tono F0 y PL-BERT)
text_aligner = load_ASR_models(f"/kaggle/working/StyleTTS2/{config['ASR_path']}", f"/kaggle/working/StyleTTS2/{config['ASR_config']}")
pitch_extractor = load_F0_models(f"/kaggle/working/StyleTTS2/{config['F0_path']}")
plbert = load_plbert(f"/kaggle/working/StyleTTS2/{config['PLBERT_dir']}")

# 3. Construir la arquitectura principal de StyleTTS 2 y pasarla a GPU en modo evaluación
model_params = recursive_munch(config['model_params'])
model = build_model(model_params, text_aligner, pitch_extractor, plbert)
_ = [model[key].eval() for key in model]
_ = [model[key].to(device) for key in model]

# 4. Cargar los pesos entrenados en cada sub-red del modelo
params_whole = torch.load(model_path, map_location='cpu')
params = params_whole['net']
for key in model:
    if key in params:
        try:
            model[key].load_state_dict(params[key])
        except:
            # Limpiar prefijo 'module.' si el modelo se entrenó en multi-GPU
            new_state_dict = OrderedDict((k[7:], v) for k, v in params[key].items())
            model[key].load_state_dict(new_state_dict, strict=False)
_ = [model[key].eval() for key in model]

# 5. Inicializar transformador a espectrograma Mel y fonemizador en inglés
to_mel = torchaudio.transforms.MelSpectrogram(n_mels=80, n_fft=2048, win_length=1200, hop_length=300)
mean, std = -4, 4
global_phonemizer = phonemizer.backend.EspeakBackend(language='en-us', preserve_punctuation=True, with_stress=True)
textclenaer = TextCleaner()

def preprocess(wave):
    """Convierte la onda de audio en un espectrograma Mel normalizado."""
    wave_tensor = torch.from_numpy(wave).float()
    mel_tensor = to_mel(wave_tensor)
    return (torch.log(1e-5 + mel_tensor.unsqueeze(0)) - mean) / std

@torch.no_grad()
def extract_style_vector(wav_path):
    """Extrae el vector latente z de 256 dimensiones (128 acústicas + 128 prosódicas) de un audio."""
    wave, sr = librosa.load(wav_path, sr=24000)
    audio, _ = librosa.effects.trim(wave, top_db=30)  # Recortar silencios al inicio y final
    mel_tensor = preprocess(audio).to(device)
    
    # Extraer mitad acústica (timbre) y mitad prosódica (ritmo/entonación)
    ref_s = model.style_encoder(mel_tensor.unsqueeze(1))
    ref_p = model.predictor_encoder(mel_tensor.unsqueeze(1))
    
    # Concatenar ambos en un único vector z de 256 dimensiones
    z = torch.cat([ref_s, ref_p], dim=1)
    return z.squeeze(0).cpu().numpy()

@torch.no_grad()
def synthesize_from_z(text, z_np):
    """Genera una onda de audio a partir de un texto y un vector latente z modificado."""
    # Convertir texto a fonemas y tokens numéricos
    ps = global_phonemizer.phonemize([text.strip()])
    ps = " ".join(word_tokenize(ps[0]))
    tokens = textclenaer(ps)
    tokens.insert(0, 0)
    tokens = torch.LongTensor(tokens).to(device).unsqueeze(0)
    
    # Preparar nuestro vector z como entrada de condicionamiento
    ref_s = torch.from_numpy(z_np).float().to(device).unsqueeze(0)
    input_lengths = torch.LongTensor([tokens.shape[-1]]).to(tokens.device)
    text_mask = length_to_mask(input_lengths).to(tokens.device)

    # Codificar texto y extraer características lingüísticas con PL-BERT
    t_en = model.text_encoder(tokens, input_lengths, text_mask)
    bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
    d_en = model.bert_encoder(bert_dur).transpose(-1, -2) 

    # Separar el vector z en su mitad prosódica (128:) y acústica (:128)
    s = ref_s[:, 128:]
    ref = ref_s[:, :128]

    # Predecir duración de cada fonema y alinear texto con audio
    d = model.predictor.text_encoder(d_en, s, input_lengths, text_mask)
    x, _ = model.predictor.lstm(d)
    duration = torch.sigmoid(model.predictor.duration_proj(x)).sum(axis=-1)
    pred_dur = torch.round(duration.squeeze()).clamp(min=1)

    pred_aln_trg = torch.zeros(int(input_lengths.item()), int(pred_dur.sum().data))
    c_frame = 0
    for i in range(pred_aln_trg.size(0)):
        pred_aln_trg[i, c_frame:c_frame + int(pred_dur[i].data)] = 1
        c_frame += int(pred_dur[i].data)

    # Predecir contorno de tono (F0) y energía, y sintetizar la forma de onda final
    en = (d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(device))
    F0_pred, N_pred = model.predictor.F0Ntrain(en, s)
    asr = (t_en @ pred_aln_trg.unsqueeze(0).to(device))
    out = model.decoder(asr, F0_pred, N_pred, ref.squeeze().unsqueeze(0))
    
    return out.squeeze().cpu().numpy()[..., :-50]

In [ ]:
import IPython.display as ipd

# 1. Crear carpeta de salida para los audios de prueba
out_dir = "/kaggle/working/hito_viabilidad_audios"
os.makedirs(out_dir, exist_ok=True)
df_spk = pd.read_csv("/kaggle/working/data_splits/vctk_speakers_split.csv")

# 2. Seleccionar una locutora (F) y un locutor (M) del conjunto de TEST
spk_f = df_spk[(df_spk["split"] == "test") & (df_spk["GENDER"] == "F")].iloc[0][
    "ID"
]
spk_m = df_spk[(df_spk["split"] == "test") & (df_spk["GENDER"] == "M")].iloc[0][
    "ID"
]

# 3. Extraer y promediar los vectores z de 5 audios de cada uno para obtener una representación limpia
wavs_f = sorted(glob.glob(f"{audio_base}/{spk_f}/*.*"))[:5]
wavs_m = sorted(glob.glob(f"{audio_base}/{spk_m}/*.*"))[:5]

z_f = np.mean([extract_style_vector(w) for w in wavs_f], axis=0)
z_m = np.mean([extract_style_vector(w) for w in wavs_m], axis=0)

test_sentence = "Interpretable control of vocal identity allows designing new synthetic voices without cloning real individuals."

# 4. Generar y guardar las voces base originales reconstruidas
sf.write(
    f"{out_dir}/1_base_{spk_f}_F.wav",
    synthesize_from_z(test_sentence, z_f),
    24000,
)
sf.write(
    f"{out_dir}/2_base_{spk_m}_M.wav",
    synthesize_from_z(test_sentence, z_m),
    24000,
)

# 5. Generar y guardar una interpolación lineal al 50% entre ambas voces
z_interp = 0.5 * z_f + 0.5 * z_m
sf.write(
    f"{out_dir}/3_interpolacion_50_50.wav",
    synthesize_from_z(test_sentence, z_interp),
    24000,
)

# 6. Desplazar progresivamente la voz femenina hacia la masculina (alpha = 0.5, 1.0, 1.5)
d_vec = z_m - z_f
for alpha in [0.5, 1.0, 1.5]:
    z_shifted = z_f + alpha * d_vec
    sf.write(
        f"{out_dir}/4_desplazamiento_alpha_{alpha}.wav",
        synthesize_from_z(test_sentence, z_shifted),
        24000,
    )

print(
    f"Hito de viabilidad completado con éxito. Audios guardados en: {out_dir}"
)
# Mostrar reproductor directo de la interpolación para verificar que suena bien
ipd.display(ipd.Audio(f"{out_dir}/3_interpolacion_50_50.wav", rate=24000))

In [ ]:
import shutil

# Comprimir la carpeta de audios en un archivo .zip dentro de /kaggle/working
zip_path = shutil.make_archive(
    "/kaggle/working/hito_viabilidad_audios",
    "zip",
    "/kaggle/working/hito_viabilidad_audios",
)
print(f"ZIP creado en: {zip_path}")